In [0]:
from pyspark.sql import functions as F

# Read the Bronze Delta table
bronze_df = spark.read.table("bronze_orders")

print("Bronze record count:", bronze_df.count())

# 1. Convert order_date to a date
silver_df = bronze_df.withColumn(
    "order_date",
    F.to_date(F.col("order_date"), "yyyy-MM-dd")
)

# 2. Remove records with missing mandatory fields
silver_df = silver_df.filter(
    F.col("order_id").isNotNull() &
    F.col("customer_id").isNotNull() &
    F.col("product").isNotNull() &
    F.col("quantity").isNotNull() &
    F.col("price").isNotNull() &
    F.col("order_date").isNotNull()
)

# 3. Keep one record per order_id
silver_df = silver_df.dropDuplicates(["order_id"])

# 4. Apply business rules
silver_df = silver_df.filter(
    (F.col("quantity") > 0) &
    (F.col("price") >= 0)
)

# 5. Calculate total order value
silver_df = silver_df.withColumn(
    "total_amount",
    F.round(F.col("quantity") * F.col("price"), 2)
)

# Display the transformed data
display(silver_df)

In [0]:
silver_table = "silver_orders"

silver_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(silver_table)

print("Silver record count:", spark.read.table(silver_table).count())

In [0]:
display(spark.read.table("silver_orders"))